# News Headline Classification — DistilBERT

Fine-tunes DistilBERT to classify URL-derived news headlines as **Fox News** or **NBC News**, and writes the trained weights to `model.pt` (loadable by `model.py`).

Run on Colab GPU. Upload `combined_url_balanced_8k.csv` to the working directory before running.

## 1. Install dependencies

In [6]:
!pip install -q transformers pandas openpyxl scikit-learn tqdm

## 2. Imports and device check

In [7]:
import os, re
from urllib.parse import urlparse, unquote

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

import warnings; warnings.filterwarnings('ignore')

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  |  device = {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
elif device.type == 'cpu':
    print('  WARNING: no GPU detected. Switch the Colab runtime to GPU before training.')

PyTorch 2.10.0+cu128  |  device = cuda
  GPU: Tesla T4


## 3. Configuration

In [8]:
class Cfg:
    DATA_PATH    = 'combined_url_balanced_8k.csv'
    OUT_WEIGHTS  = 'model.pt'

    BACKBONE     = 'distilbert-base-uncased'
    MAX_LENGTH   = 128
    NUM_CLASSES  = 2

    BATCH_SIZE   = 32
    LR           = 2e-5
    EPOCHS       = 3
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    DROPOUT      = 0.1
    GRAD_CLIP    = 1.0

    VAL_FRACTION = 0.15
    SEED         = 42

    LABEL_TO_ID  = {'foxnews': 0, 'nbcnews': 1}
    ID_TO_LABEL  = {0: 'foxnews', 1: 'nbcnews'}

print(f'data       = {Cfg.DATA_PATH}')
print(f'epochs     = {Cfg.EPOCHS}')
print(f'batch size = {Cfg.BATCH_SIZE}')
print(f'seed       = {Cfg.SEED}')

data       = combined_url_balanced_8k.csv
epochs     = 3
batch size = 32
seed       = 42


## 4. URL processing

Slug extraction matches `preprocess.py` so the text the model trains on is the same text the grader feeds it at evaluation time.

In [9]:
def label_from_url(url):
    u = str(url).lower()
    if 'foxnews.com' in u:
        return 'foxnews'
    if 'nbcnews.com' in u:
        return 'nbcnews'
    return None


def text_from_url(url):
    """Walk the URL path right-to-left; return the first segment that looks like a real slug."""
    try:
        parsed = urlparse(str(url))
        segments = [s for s in unquote(parsed.path).split('/') if s]
        if not segments:
            return ''

        slug = ''
        for seg in reversed(segments):
            # NBC appends an article ID like '...-rcna134498' — skip that segment
            if 'rcna' in seg.lower():
                continue
            # skip purely numeric / mostly-numeric segments
            if re.match(r'^[a-z]*\d+$', seg.lower()):
                continue
            # skip segments too short to be a headline
            if len(seg) < 5:
                continue
            slug = seg
            break

        if not slug:
            slug = segments[-1]

        text = slug.replace('-', ' ').replace('_', ' ')
        text = re.sub(r'\brcna\b', '', text, flags=re.IGNORECASE)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip().lower()
    except Exception:
        return ''


_demo = 'https://www.foxnews.com/world/australian-senator-wears-burqa-after-move'
print(f'URL  : {_demo}')
print(f'label: {label_from_url(_demo)}')
print(f'text : {text_from_url(_demo)}')

URL  : https://www.foxnews.com/world/australian-senator-wears-burqa-after-move
label: foxnews
text : australian senator wears burqa after move


## 5. Load and preprocess data

In [10]:
if Cfg.DATA_PATH.lower().endswith(('.xlsx', '.xls')):
    df = pd.read_excel(Cfg.DATA_PATH)
else:
    df = pd.read_csv(Cfg.DATA_PATH)

url_col = next((c for c in df.columns if c.lower() in {'url', 'urls', 'link', 'links'}), df.columns[0])
print(f'rows: {len(df)}  url column: {url_col!r}')

texts, labels = [], []
skipped = 0
for u in tqdm(df[url_col], desc='processing URLs'):
    if pd.isna(u):
        skipped += 1; continue
    lab = label_from_url(u)
    if lab is None:
        skipped += 1; continue
    txt = text_from_url(u)
    if not txt:
        skipped += 1; continue
    texts.append(txt); labels.append(lab)

print(f'\nkept    : {len(texts)}')
print(f'skipped : {skipped}')
print(f'foxnews : {labels.count("foxnews")}')
print(f'nbcnews : {labels.count("nbcnews")}')

rows: 8000  url column: 'url'


processing URLs:   0%|          | 0/8000 [00:00<?, ?it/s]


kept    : 8000
skipped : 0
foxnews : 4000
nbcnews : 4000


## 6. Train / validation split

In [11]:
np.random.seed(Cfg.SEED)
torch.manual_seed(Cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Cfg.SEED)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=Cfg.VAL_FRACTION,
    random_state=Cfg.SEED,
    stratify=labels,
)

print(f'train: {len(train_texts)}')
print(f'val  : {len(val_texts)}')

train: 6800
val  : 1200


## 7. PyTorch dataset

In [12]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.y = [Cfg.LABEL_TO_ID[l] for l in labels]
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.y[i], dtype=torch.long),
        }


tokenizer = DistilBertTokenizerFast.from_pretrained(Cfg.BACKBONE)
train_ds = NewsDataset(train_texts, train_labels, tokenizer, Cfg.MAX_LENGTH)
val_ds   = NewsDataset(val_texts,   val_labels,   tokenizer, Cfg.MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=Cfg.BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=Cfg.BATCH_SIZE, shuffle=False)

print(f'train batches: {len(train_loader)}')
print(f'val   batches: {len(val_loader)}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

train batches: 213
val   batches: 38


## 8. Model architecture

DistilBERT encoder → `[CLS]` vector → Dropout → Linear(768, 2). Mirrors `model.py` so the saved `state_dict` loads cleanly there.

In [13]:
class NewsClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = DistilBertModel.from_pretrained(Cfg.BACKBONE)
        self.dropout    = nn.Dropout(Cfg.DROPOUT)
        self.classifier = nn.Linear(self.bert.config.hidden_size, Cfg.NUM_CLASSES)

    def forward(self, input_ids, attention_mask):
        h = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(h))


model = NewsClassifier().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


parameters: 66,364,418


## 9. Optimizer and scheduler

In [14]:
optimizer = AdamW(model.parameters(), lr=Cfg.LR, weight_decay=Cfg.WEIGHT_DECAY)

total_steps  = len(train_loader) * Cfg.EPOCHS
warmup_steps = int(total_steps * Cfg.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'total steps : {total_steps}')
print(f'warmup steps: {warmup_steps}')

total steps : 639
warmup steps: 63


## 10. Training loop

In [15]:
loss_fn = nn.CrossEntropyLoss()

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss = 0.0
    preds, trues = [], []
    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    desc = 'train' if train_mode else 'val'
    with ctx:
        for batch in tqdm(loader, desc=desc, leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            y              = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss   = loss_fn(logits, y)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), Cfg.GRAD_CLIP)
                optimizer.step(); scheduler.step()

            total_loss += loss.item()
            preds.extend(logits.argmax(-1).cpu().tolist())
            trues.extend(y.cpu().tolist())

    return total_loss / len(loader), accuracy_score(trues, preds), preds, trues


best_acc = 0.0
for epoch in range(Cfg.EPOCHS):
    print(f'\nepoch {epoch + 1}/{Cfg.EPOCHS}')
    train_loss, train_acc, _, _ = run_epoch(train_loader, train_mode=True)
    val_loss,   val_acc,   _, _ = run_epoch(val_loader,   train_mode=False)
    print(f'  train loss {train_loss:.4f}  acc {train_acc:.4f}')
    print(f'  val   loss {val_loss:.4f}  acc {val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), Cfg.OUT_WEIGHTS)
        print(f'  saved {Cfg.OUT_WEIGHTS} (val acc {val_acc:.4f})')

print(f'\nbest val accuracy: {best_acc:.4f}')


epoch 1/3


train:   0%|          | 0/213 [00:00<?, ?it/s]

val:   0%|          | 0/38 [00:00<?, ?it/s]

  train loss 0.1344  acc 0.9332
  val   loss 0.0036  acc 0.9983
  saved model.pt (val acc 0.9983)

epoch 2/3


train:   0%|          | 0/213 [00:00<?, ?it/s]

val:   0%|          | 0/38 [00:00<?, ?it/s]

  train loss 0.0017  acc 0.9999
  val   loss 0.0058  acc 0.9983

epoch 3/3


train:   0%|          | 0/213 [00:00<?, ?it/s]

val:   0%|          | 0/38 [00:00<?, ?it/s]

  train loss 0.0002  acc 1.0000
  val   loss 0.0025  acc 0.9992
  saved model.pt (val acc 0.9992)

best val accuracy: 0.9992


## 11. Final evaluation

In [16]:
model.load_state_dict(torch.load(Cfg.OUT_WEIGHTS, map_location=device))
_, final_acc, preds, trues = run_epoch(val_loader, train_mode=False)

print(f'val accuracy: {final_acc:.4f}\n')
print(classification_report(trues, preds, target_names=['foxnews', 'nbcnews']))

size_mb = os.path.getsize(Cfg.OUT_WEIGHTS) / (1024 * 1024)
print(f'saved weights: {Cfg.OUT_WEIGHTS}  ({size_mb:.1f} MB)')

val:   0%|          | 0/38 [00:00<?, ?it/s]

val accuracy: 0.9992

              precision    recall  f1-score   support

     foxnews       1.00      1.00      1.00       600
     nbcnews       1.00      1.00      1.00       600

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200

saved weights: model.pt  (253.2 MB)
